# `swift_too` module

## Async `get()` pattern with `asyncio.gather`

### API version = 1.2, `swifttools` version 4.0

#### Author: Jamie A. Kennea (Penn State)

`swifttools` version 4.0 introduces real `asyncio` support with the
introduction of new async `get()` and `post()` methods. These replace the
`submit()` and `queue()` methods. Note that `queue()` is performed using
threading so is not true async, however `get()` and `post()` return coroutines
that need to be awaited.

This notebook demonstrates the recommended async `get()` workflow for request
classes that support coroutine-based retrieval.

The examples use `VisQuery` as a concrete class, but the same pattern applies
to other classes that expose an async `get()` method:

1. Build request objects.
2. Run one request directly with `await`.
3. Run many requests concurrently with `asyncio.gather`.
4. Optionally limit concurrency with a semaphore.
5. Compare sequential and concurrent execution time.


In [ ]:
import asyncio
import time

from swifttools.swift_too import VisQuery
from swifttools.swift_too.swift.resolve import Resolve

## Single async `get()` request

Here we demonstrate how `async get()` works with `VisQuery`. Note that
`autosubmit=False` is manditory with async requests, as otherwise the class
will perform a submit on instantiation. Here we simply create the query
coroutine, and then `await` it. The resulting value of `q` will be the same as
before.  

In [ ]:
q = VisQuery(name="Crab", length=1, hires=False, autosubmit=False)
if await q.get():
    print(q)
else:
    print(f"Error: {q.status.errors}")

## Concurrent gather: one coroutine per target

This demonstrates the use of async to perform multiple API tasks asynchronously
using `asyncio.gather`. `asyncio.gather` schedules all target coroutines
together and waits for them as a group.

First is creates a list of queries, then in `asyncio.gather` executes
them asyncronously using the `get()` method. 

This demonstration uses `return_exceptions=True` so one failing request does
not cancel the full batch. 

In this test, we do `SwiftResolve` (AKA `Resolve`) for 

In [ ]:
targets = ["Crab", "Vega", "Cyg X-1", "M31", "Turkey Nebula"]
queries = [Resolve(name=target, autosubmit=False) for target in targets]

results = await asyncio.gather(*(query.get() for query in queries), return_exceptions=True)

for query, result in zip(queries, results):
    if isinstance(result, Exception):
        print(f"error resolving {query.name}: {result}")
        continue

    print(
        f"{query.name}: ok={result}, status={query.status.status}, "
        f"ra={query.ra}, dec={query.dec}, resolver={query.resolver}"
    )

## Bounded concurrency with a semaphore

The Swift API is a shared resource, and as such should not be hammered with
thousands of async requests. Although rate-limiting is implemented, we prefer
that users self limit themslves. One way to do this is to use a a semaphore to
cap the simultaneous in-flight requests while keeping the gather pattern.

In [ ]:
targets = ["Crab", "Vega", "Cyg X-1", "M31"]
queries = [VisQuery(name=name, length=1, hires=False, autosubmit=False) for name in targets]

results = await asyncio.gather(*(q.get() for q in queries), return_exceptions=True)

for target, q, result in zip(targets, queries, results):
    if isinstance(result, Exception):
        print(f"error: {target}: {result}")
    else:
        print({"target": target, "ok": result, "status": q.status.status, "windows": len(q.windows)})

## Timing comparison: sequential await vs concurrent gather

Performing tasks asynchronously will naturally decrease latency in code, as
tasks are run concurrently rather than in series. This means that instead of
waiting for the result of the first API call to return before moving to the
next, the requests are submitted sequentially. This has the impact of
significantly reducing overall latency, as show below.

In [ ]:
targets_timing = ["Crab", "Vega", "Cyg X-1"]


async def fetch_vis(name: str) -> dict[str, object]:
    q = VisQuery(name=name, length=1, hires=False, autosubmit=False)
    ok = await q.get()
    return {"target": name, "ok": ok, "status": q.status.status, "windows": len(q.windows)}


t0 = time.perf_counter()
seq = [await fetch_vis(name) for name in targets_timing]
t1 = time.perf_counter()

t2 = time.perf_counter()
conc = await asyncio.gather(*(fetch_vis(name) for name in targets_timing), return_exceptions=True)
t3 = time.perf_counter()

print(f"sequential: {t1 - t0:.2f}s")
print(f"concurrent: {t3 - t2:.2f}s")
print("sequential results:", seq)
print("concurrent results:", conc)